# Generación de resultados por modelo

Ejecutar una vez por modelo cambiando únicamente `MODEL_KEY` en la celda 1.  
Guarda en Drive: `z0.pt`, imágenes y `stats.json` para cada modelo.

**Flujo recomendado:**
1. Ejecuta con `MODEL_KEY = "sd15"` → guarda en `modelos/resultados/sd15/`
2. *Entorno de ejecución → Reiniciar sesión*
3. Repite con `"sd21"` y `"sdxl"`
4. Descarga `modelos/resultados/` y abre `comparativa_inicial.ipynb` en local (no requiere GPU)

**Prerequisito:** ejecutar `TFM_setup.ipynb` al inicio de cada sesión.

---
## 0 — Entorno: Drive, paths y dependencias

In [ ]:
from google.colab import drive
import gc, os, sys, torch

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = "/content/drive/MyDrive/TFM/"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.environ["HF_HOME"] = "/content/.cache/huggingface"

assert torch.cuda.is_available(), "GPU no disponible. Activa T4 en Entorno de ejecución."
print(f"GPU: {torch.cuda.get_device_name(0)}  —  "
      f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"HF_HOME  : {os.environ['HF_HOME']}")
print(f"Proyecto : {PROJECT_ROOT}")

In [ ]:
# Instalación de dependencias
req = os.path.join(PROJECT_ROOT, "requirements_colab.txt")
!pip install -q -U -r {req}
print("Dependencias instaladas.")

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = ""  # <-- pega tu token hf_...

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN vacío. Rellena la variable con tu token de "
        "https://huggingface.co/settings/tokens (tipo Read) "
        "y vuelve a ejecutar esta celda."
    )

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
print("Autenticado en Hugging Face.")

---
## 1 — Configuración

**Único parámetro a cambiar entre sesiones.**

In [ ]:
# ── CAMBIA ESTE VALOR ENTRE SESIONES ──────────────────────────────────────────
MODEL_KEY = "sd15"   # "sd15" | "sd21" | "sdxl"
# ──────────────────────────────────────────────────────────────────────────────

PROMPT         = "a red apple on a wooden table, studio lighting, photorealistic"
SEED           = 42
# SDXL usa 30 pasos porque su proceso de denoising converge más rápido gracias
# al UNet de mayor capacidad y al condicionamiento textual dual. SD 1.5 y SD 2.1
# necesitan 50 pasos para alcanzar calidad comparable. Ambos valores son los
# estándar de la literatura para cada arquitectura.
NUM_STEPS      = 30 if MODEL_KEY == "sdxl" else 50
GUIDANCE_SCALE = 7.5

OUTPUT_DIR = os.path.join(PROJECT_ROOT, "data", "fase1", MODEL_KEY)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Modelo     : {MODEL_KEY}")
print(f"Steps      : {NUM_STEPS}")
print(f"Salidas    : {OUTPUT_DIR}")

---
## 2 — Tabla comparativa de arquitecturas

Inspección estática del registro de modelos (no carga pesos).

In [ ]:
from models.loader import SUPPORTED_MODELS
import pandas as pd

rows = []
for key, cfg in SUPPORTED_MODELS.items():
    res  = cfg["default_res"]
    hw   = res // cfg["vae_scale_factor"]
    rows.append({
        "Clave":         key,
        "Modelo":        cfg["model_id"].split("/")[-1],
        "Encoder texto": cfg["text_encoder"],
        "Res. nativa":   f"{res}×{res}",
        "Forma latente": f"4 × {hw} × {hw}",
        "Dim. latente":  4 * hw * hw,
        "Params (B)":    cfg["params_B"],
    })

df = pd.DataFrame(rows).set_index("Clave")
print(df.to_string())

---
## 3 — Carga del modelo

In [ ]:
from models import load_model

model = load_model(MODEL_KEY)
print(model)
print(f"VRAM usada: {torch.cuda.memory_allocated() / 1024**3:.1f} GB")

---
## 4 — Generación y captura de z₀

In [ ]:
# Generación de la imagen de prueba con captura del latente z₀.
# z₀ es el tensor latente denoised final, justo antes de ser decodificado por el VAE.
# Es el objeto central de análisis del TFM: su estructura geométrica y semántica
# refleja cómo el modelo ha organizado el contenido visual durante el denoising.
# La captura se realiza mediante un hook sobre vae.decode en models/loader.py.
image, z0 = model.generate(
    PROMPT,
    seed=SEED,
    num_inference_steps=NUM_STEPS,
    guidance_scale=GUIDANCE_SCALE,
)

print(f"Imagen generada : {image.size}")
print(f"z0 shape        : {z0.shape}")
print(f"z0 dtype        : {z0.dtype}")
print(f"z0 norma (media): {z0.float().norm(dim=1).mean().item():.4f}")
image

---
## 5 — Round-trip VAE (encode → decode)

In [ ]:
# Verificación del ciclo encode→decode del VAE (round-trip).
# encode_image() comprime la imagen al espacio latente usando la MEDIA de la
# distribución posterior (sin ruido), lo que garantiza determinismo.
# decode_latent() invierte el proceso. El MAE resultante cuantifica la fidelidad
# del autoencoder: valores <5/255 son subperceptuales a resolución nativa.
# Este análisis valida que el VAE de cada modelo codifica información suficiente
# para reconstruir la imagen de forma fiel desde el espacio z₀.
import matplotlib.pyplot as plt
import numpy as np

z_enc = model.encode_image(image)
recon = model.decode_latent(z_enc)

diff = np.abs(np.array(image).astype(float) - np.array(recon).astype(float))
mae  = float(diff.mean())
print(f"Error absoluto medio (encode→decode): {mae:.2f} / 255")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(image); axes[0].set_title("Original generada")
axes[1].imshow(recon);  axes[1].set_title("Reconstruida (encode→decode)")
axes[2].imshow((diff * 5).clip(0, 255).astype(np.uint8), cmap="hot")
axes[2].set_title("Diferencia ×5")
for ax in axes: ax.axis("off")
plt.suptitle(f"{MODEL_KEY.upper()} — Round-trip VAE", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "roundtrip.png"), dpi=150)
plt.show()

---
## 6 — Guardar resultados en Drive

In [ ]:
# Persistencia de resultados en Google Drive.
# z₀ se guarda en float32 (en lugar del float16 de inferencia) para garantizar
# compatibilidad al cargarlo en el entorno local y evitar pérdidas de precisión
# en operaciones de análisis como PCA y cálculo de distancias euclídeas.
# stats.json registra los metadatos de la sesión para reproducibilidad.
import json, datetime

# z0 en float32 para mayor compatibilidad al cargar en local
torch.save(z0.cpu().float(), os.path.join(OUTPUT_DIR, "z0.pt"))

image.save(os.path.join(OUTPUT_DIR, "imagen_generada.png"))
recon.save(os.path.join(OUTPUT_DIR, "imagen_reconstruida.png"))

z_f = z0.float()
stats = {
    "model_key":    MODEL_KEY,
    "fecha":        datetime.datetime.now().isoformat(),
    "prompt":       PROMPT,
    "seed":         SEED,
    "num_steps":    NUM_STEPS,
    "latent_shape": list(z0.shape),
    "latent_dim":   int(z0.numel()),
    "z0_mean":      round(z_f.mean().item(), 6),
    "z0_std":       round(z_f.std().item(), 6),
    "z0_norm":      round(z_f.norm().item(), 4),
    "vae_mae":      round(mae, 4),
}
with open(os.path.join(OUTPUT_DIR, "stats.json"), "w") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print(f"Resultados guardados en: {OUTPUT_DIR}")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, fname)) / 1024
    print(f"  {fname:<35} {size_kb:>8.1f} KB")